# Pipeline Spark — Load, Full Dataset, Join Dataset Yelp

Notebook ini memproses dataset mentah Yelp menjadi satu tabel tergabung siap pakai untuk EDA dan modeling.

**Dataset input** (di `data/raw/`):
- `yelp_academic_dataset_review.json` (~5 GB, ~7 juta ulasan)
- `yelp_academic_dataset_business.json` (~120 MB)
- `yelp_academic_dataset_user.json` (~3 GB)

**Output** (di `data/processed/`):
- `reviews_sampled.parquet` — seluruh 6.990.280 ulasan gabungan (review + business + user)
- `business_clean.parquet`, `user_clean.parquet`

Urutan: load JSON → filter kolom → label encoding → join 3 tabel → simpan Parquet.

## 0. Import Library

In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

import pandas as pd
import numpy as np

print("Import selesai.")

Import selesai.


## 1. Inisialisasi SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("YelpSentimentPipeline")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "4g")
    .config("spark.local.dir", "D:/spark-temp")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.ui.showConsoleProgress", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"SparkSession aktif: {spark.version}")
print(f"App Name         : {spark.sparkContext.appName}")
print(f"Master           : {spark.sparkContext.master}")

SparkSession aktif: 3.3.2
App Name         : YelpSentimentPipeline
Master           : local[*]


## 2. Definisi Path

In [3]:
BASE_DIR = r"D:\big-data-ai-sentiment"

PATH_REVIEW   = os.path.join(BASE_DIR, "data", "raw", "yelp_academic_dataset_review.json")
PATH_BUSINESS = os.path.join(BASE_DIR, "data", "raw", "yelp_academic_dataset_business.json")
PATH_USER     = os.path.join(BASE_DIR, "data", "raw", "yelp_academic_dataset_user.json")

PATH_PROCESSED    = os.path.join(BASE_DIR, "data", "processed")
# Spark menulis ke folder sementara dulu, baru dicopy ke file tunggal
PATH_REVIEWS_OUT  = os.path.join(PATH_PROCESSED, "_reviews_sampled_spark")
PATH_BUSINESS_OUT = os.path.join(PATH_PROCESSED, "business_clean.parquet")
PATH_USER_OUT     = os.path.join(PATH_PROCESSED, "user_clean.parquet")
FINAL_CSV         = os.path.join(PATH_PROCESSED, "reviews_sampled.csv")

os.makedirs(PATH_PROCESSED, exist_ok=True)

for label, path in [("Review", PATH_REVIEW), ("Business", PATH_BUSINESS), ("User", PATH_USER)]:
    ukuran = os.path.getsize(path) / (1024**3)
    print(f"{label:10s}: {path} ({ukuran:.2f} GB)")

Review    : D:\big-data-ai-sentiment\data\raw\yelp_academic_dataset_review.json (4.98 GB)
Business  : D:\big-data-ai-sentiment\data\raw\yelp_academic_dataset_business.json (0.11 GB)
User      : D:\big-data-ai-sentiment\data\raw\yelp_academic_dataset_user.json (3.13 GB)


## 3. Load & Proses Dataset Review

In [4]:
print("Membaca dataset review...")
df_review_raw = spark.read.json(PATH_REVIEW)

print(f"Total baris : {df_review_raw.count():,}")
print(f"Total kolom : {len(df_review_raw.columns)}")
df_review_raw.printSchema()

Membaca dataset review...
Total baris : 6,990,280
Total kolom : 9
root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)



In [5]:
KOLOM_REVIEW = ["review_id", "user_id", "business_id", "stars", "text", "date"]

df_review = df_review_raw.select(KOLOM_REVIEW)

print(f"Kolom tersisa: {df_review.columns}")
df_review.show(3, truncate=80)

Kolom tersisa: ['review_id', 'user_id', 'business_id', 'stars', 'text', 'date']
+----------------------+----------------------+----------------------+-----+--------------------------------------------------------------------------------+-------------------+
|             review_id|               user_id|           business_id|stars|                                                                            text|               date|
+----------------------+----------------------+----------------------+-----+--------------------------------------------------------------------------------+-------------------+
|KU_O5udG6zpxOg-VcAEodg|mh_-eMZ6K5RLWhZyISBhwA|XQfwVwDr-v0ZS3_CbbE5Xw|  3.0|If you decide to eat here, just be aware it is going to take about 2 hours fr...|2018-07-07 22:09:11|
|BiTunyQ73aT9WBnpR9DZGw|OyoGAe7OKpv6SyGZT5g77Q|7ATYjTIgM3jUlt4UM3IypQ|  5.0|I've taken a lot of spin classes over the years, and nothing compares to the ...|2012-01-03 15:28:18|
|saUsX_uimxRlCVr67Z4Jig|8g_iMt

In [6]:
baris_sebelum = df_review.count()

df_review = df_review.dropna(subset=["review_id", "user_id", "business_id", "stars", "text", "date"])
df_review = df_review.filter(F.trim(F.col("text")) != "")

baris_sesudah = df_review.count()
print(f"Baris sebelum : {baris_sebelum:,}")
print(f"Baris sesudah : {baris_sesudah:,}")
print(f"Dihapus       : {baris_sebelum - baris_sesudah:,}")

Baris sebelum : 6,990,280
Baris sesudah : 6,990,280
Dihapus       : 0


In [7]:
# Label encoding: stars → sentimen
# 1-2 bintang = Negatif (0), 3 = Netral (1), 4-5 = Positif (2)
df_review = df_review.withColumn(
    "sentiment",
    F.when(F.col("stars") <= 2, 0)
     .when(F.col("stars") == 3, 1)
     .otherwise(2)
     .cast(IntegerType())
)

print("Distribusi sentimen sebelum sampling:")
df_review.groupBy("sentiment") \
         .count() \
         .withColumn("persen", F.round(F.col("count") / baris_sesudah * 100, 2)) \
         .orderBy("sentiment") \
         .show()

df_review.show(5, truncate=60)

Distribusi sentimen sebelum sampling:
+---------+-------+------+
|sentiment|  count|persen|
+---------+-------+------+
|        0|1613801| 23.09|
|        1| 691934|   9.9|
|        2|4684545| 67.02|
+---------+-------+------+

+----------------------+----------------------+----------------------+-----+------------------------------------------------------------+-------------------+---------+
|             review_id|               user_id|           business_id|stars|                                                        text|               date|sentiment|
+----------------------+----------------------+----------------------+-----+------------------------------------------------------------+-------------------+---------+
|KU_O5udG6zpxOg-VcAEodg|mh_-eMZ6K5RLWhZyISBhwA|XQfwVwDr-v0ZS3_CbbE5Xw|  3.0|If you decide to eat here, just be aware it is going to t...|2018-07-07 22:09:11|        1|
|BiTunyQ73aT9WBnpR9DZGw|OyoGAe7OKpv6SyGZT5g77Q|7ATYjTIgM3jUlt4UM3IypQ|  5.0|I've taken a lot of spin

## 4. Gunakan Full Dataset (Tanpa Sampling)

In [8]:
# Pakai seluruh data tanpa sampling
df_sampled    = df_review
total_sampled = baris_sesudah

print(f"Total baris digunakan : {total_sampled:,}")
print("Distribusi sentimen (full dataset):")
df_sampled.groupBy("sentiment") \
          .count() \
          .withColumn("persen", F.round(F.col("count") / total_sampled * 100, 2)) \
          .orderBy("sentiment") \
          .show()

Total baris digunakan : 6,990,280
Distribusi sentimen (full dataset):
+---------+-------+------+
|sentiment|  count|persen|
+---------+-------+------+
|        0|1613801| 23.09|
|        1| 691934|   9.9|
|        2|4684545| 67.02|
+---------+-------+------+



In [9]:
print("Distribusi sentimen full dataset (konfirmasi):")
df_sampled.groupBy("sentiment") \
          .count() \
          .withColumn("persen", F.round(F.col("count") / total_sampled * 100, 2)) \
          .orderBy("sentiment") \
          .show()

Distribusi sentimen full dataset (konfirmasi):
+---------+-------+------+
|sentiment|  count|persen|
+---------+-------+------+
|        0|1613801| 23.09|
|        1| 691934|   9.9|
|        2|4684545| 67.02|
+---------+-------+------+



## 5. Load & Proses Dataset Business

In [10]:
print("Membaca dataset business...")
df_business_raw = spark.read.json(PATH_BUSINESS)

print(f"Total baris : {df_business_raw.count():,}")
print(f"Total kolom : {len(df_business_raw.columns)}")
df_business_raw.printSchema()

Membaca dataset business...
Total baris : 150,346
Total kolom : 14
root
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: string (nullable = true)
 |    |-- Caters: string (nullable = true)
 |    |-- CoatCheck: string (nullable = true)
 |    |-- Corkage: string (nullable = true)
 |    |-- DietaryRestrictions: string (nullable = true)
 |    |-- DogsAllowed: string (nullable = true)
 |    |-

In [11]:
KOLOM_BUSINESS = ["business_id", "name", "city", "state", "categories", "stars", "review_count"]

df_business = df_business_raw.select(KOLOM_BUSINESS)
df_business = df_business.dropna(subset=["business_id", "city", "categories"])

# rename agar tidak tabrakan dengan kolom stars/review_count dari tabel review
df_business = df_business \
    .withColumnRenamed("stars", "business_stars") \
    .withColumnRenamed("review_count", "business_review_count")

print(f"Baris business: {df_business.count():,}")
df_business.show(3, truncate=60)

Baris business: 150,243
+----------------------+------------------------+-------------+-----+------------------------------------------------------------+--------------+---------------------+
|           business_id|                    name|         city|state|                                                  categories|business_stars|business_review_count|
+----------------------+------------------------+-------------+-----+------------------------------------------------------------+--------------+---------------------+
|Pns2l4eNsfO8kk83dixA6A|Abby Rappoport, LAC, CMQ|Santa Barbara|   CA|Doctors, Traditional Chinese Medicine, Naturopathic/Holis...|           5.0|                    7|
|mpf3x-BjTdTEA3yCZrAYPw|           The UPS Store|       Affton|   MO|Shipping Centers, Local Services, Notaries, Mailbox Cente...|           3.0|                   15|
|tUFrWirKiKi_TAnsVWINQQ|                  Target|       Tucson|   AZ|Department Stores, Shopping, Fashion, Home & Garden, Elec...|      

## 6. Load & Proses Dataset User

In [12]:
print("Membaca dataset user...")
df_user_raw = spark.read.json(PATH_USER)

print(f"Total baris : {df_user_raw.count():,}")
print(f"Total kolom : {len(df_user_raw.columns)}")
df_user_raw.printSchema()

Membaca dataset user...
Total baris : 1,987,897
Total kolom : 22
root
 |-- average_stars: double (nullable = true)
 |-- compliment_cool: long (nullable = true)
 |-- compliment_cute: long (nullable = true)
 |-- compliment_funny: long (nullable = true)
 |-- compliment_hot: long (nullable = true)
 |-- compliment_list: long (nullable = true)
 |-- compliment_more: long (nullable = true)
 |-- compliment_note: long (nullable = true)
 |-- compliment_photos: long (nullable = true)
 |-- compliment_plain: long (nullable = true)
 |-- compliment_profile: long (nullable = true)
 |-- compliment_writer: long (nullable = true)
 |-- cool: long (nullable = true)
 |-- elite: string (nullable = true)
 |-- fans: long (nullable = true)
 |-- friends: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- name: string (nullable = true)
 |-- review_count: long (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- yelping_since: string (nullable = true)



In [13]:
KOLOM_USER = ["user_id", "review_count", "yelping_since", "average_stars", "fans"]

df_user = df_user_raw.select(KOLOM_USER)
df_user = df_user.dropna(subset=["user_id", "review_count"])

# klasifikasi user berdasarkan total review: Casual ≤10, Active 11-50, Power >50
df_user = df_user.withColumn(
    "user_type",
    F.when(F.col("review_count") <= 10, "Casual")
     .when(F.col("review_count") <= 50, "Active")
     .otherwise("Power")
)

print(f"Baris user: {df_user.count():,}")
print("\nDistribusi user_type:")
df_user.groupBy("user_type").count().orderBy("count", ascending=False).show()
df_user.show(3)

Baris user: 1,987,897

Distribusi user_type:
+---------+-------+
|user_type|  count|
+---------+-------+
|   Casual|1305941|
|   Active| 496076|
|    Power| 185880|
+---------+-------+

+--------------------+------------+-------------------+-------------+----+---------+
|             user_id|review_count|      yelping_since|average_stars|fans|user_type|
+--------------------+------------+-------------------+-------------+----+---------+
|qVc8ODYU5SZjKXVBg...|         585|2007-01-25 16:47:26|         3.91| 267|    Power|
|j14WgRoU_-2ZE1aw1...|        4333|2009-01-25 04:35:42|         3.74|3138|    Power|
|2WnXYQFK0hXEoTxPt...|         665|2008-07-25 10:41:00|         3.32|  52|    Power|
+--------------------+------------+-------------------+-------------+----+---------+
only showing top 3 rows



## 7. Join Review + Business + User

In [14]:
print("Join review + business...")
# left join supaya review yang businessnya tidak ada di dataset tetap masuk
df_joined = df_sampled.join(df_business, on="business_id", how="left")

print(f"Baris setelah join business : {df_joined.count():,}")
print(f"Kolom                       : {df_joined.columns}")

Join review + business...
Baris setelah join business : 6,990,280
Kolom                       : ['business_id', 'review_id', 'user_id', 'stars', 'text', 'date', 'sentiment', 'name', 'city', 'state', 'categories', 'business_stars', 'business_review_count']


In [15]:
print("Join review+business + user...")
df_final = df_joined.join(df_user, on="user_id", how="left")

print(f"Baris setelah join user : {df_final.count():,}")
print(f"Total kolom             : {len(df_final.columns)}")
df_final.printSchema()

Join review+business + user...
Baris setelah join user : 6,990,280
Total kolom             : 18
root
 |-- user_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- date: string (nullable = true)
 |-- sentiment: integer (nullable = false)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- business_stars: double (nullable = true)
 |-- business_review_count: long (nullable = true)
 |-- review_count: long (nullable = true)
 |-- yelping_since: string (nullable = true)
 |-- average_stars: double (nullable = true)
 |-- fans: long (nullable = true)
 |-- user_type: string (nullable = true)



In [16]:
KOLOM_FINAL = [
    "review_id", "user_id", "business_id",
    "text", "date", "stars", "sentiment",
    "name", "city", "state", "categories",
    "business_stars", "business_review_count",
    "review_count", "yelping_since", "average_stars", "fans", "user_type",
]

df_final = df_final.select(KOLOM_FINAL)

df_final.show(5, truncate=50)
print(f"\nTotal baris : {df_final.count():,}")
print(f"Total kolom : {len(df_final.columns)}")

+----------------------+----------------------+----------------------+--------------------------------------------------+-------------------+-----+---------+----------------------------+------------+-----+--------------------------------------------------+--------------+---------------------+------------+-------------------+-------------+----+---------+
|             review_id|               user_id|           business_id|                                              text|               date|stars|sentiment|                        name|        city|state|                                        categories|business_stars|business_review_count|review_count|      yelping_since|average_stars|fans|user_type|
+----------------------+----------------------+----------------------+--------------------------------------------------+-------------------+-----+---------+----------------------------+------------+-----+--------------------------------------------------+--------------+-----------------

In [17]:
# null wajar di kolom business/user karena pakai left join
print("Missing values per kolom:")
null_counts = df_final.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_final.columns
])
null_counts.show(vertical=True)

Missing values per kolom:
-RECORD 0--------------------
 review_id             | 0   
 user_id               | 0   
 business_id           | 0   
 text                  | 0   
 date                  | 0   
 stars                 | 0   
 sentiment             | 0   
 name                  | 689 
 city                  | 689 
 state                 | 689 
 categories            | 689 
 business_stars        | 689 
 business_review_count | 689 
 review_count          | 33  
 yelping_since         | 33  
 average_stars         | 33  
 fans                  | 33  
 user_type             | 33  



## 8. Simpan Output ke Disk

In [18]:
print("Menyimpan reviews_sampled ke CSV (sebagai backup)...")

df_final.coalesce(1) \
        .write \
        .option("header", True) \
        .mode("overwrite") \
        .csv(PATH_REVIEWS_OUT)

print(f"Tersimpan di: {PATH_REVIEWS_OUT}")

Menyimpan reviews_sampled ke CSV (sebagai backup)...
Tersimpan di: D:\big-data-ai-sentiment\data\processed\_reviews_sampled_spark


In [19]:
print("Menyimpan business_clean.parquet...")
df_business.coalesce(1).write.mode("overwrite").parquet(PATH_BUSINESS_OUT)
print(f"Tersimpan di: {PATH_BUSINESS_OUT}")

Menyimpan business_clean.parquet...
Tersimpan di: D:\big-data-ai-sentiment\data\processed\business_clean.parquet


In [20]:
print("Menyimpan user_clean.parquet...")
df_user.coalesce(1).write.mode("overwrite").parquet(PATH_USER_OUT)
print(f"Tersimpan di: {PATH_USER_OUT}")

Menyimpan user_clean.parquet...
Tersimpan di: D:\big-data-ai-sentiment\data\processed\user_clean.parquet


## 9. Verifikasi Output

In [21]:
import glob

csv_files = glob.glob(os.path.join(PATH_REVIEWS_OUT, "part-*.csv"))
print(f"File CSV ditemukan: {csv_files}")

df_check = pd.read_csv(csv_files[0], nrows=5)
print(f"\nKolom  : {list(df_check.columns)}")
print(f"Shape  : {df_check.shape}")
df_check

File CSV ditemukan: ['D:\\big-data-ai-sentiment\\data\\processed\\_reviews_sampled_spark\\part-00000-294d1f38-fc03-4188-b6c7-f3cdf624dad3-c000.csv']

Kolom  : ['review_id', 'user_id', 'business_id', 'text', 'date', 'stars', 'sentiment', 'name', 'city', 'state', 'categories', 'business_stars', 'business_review_count', 'review_count', 'yelping_since', 'average_stars', 'fans', 'user_type']
Shape  : (5, 18)


,review_id,user_id,business_id,text,date,stars,sentiment,name,city,state,categories,business_stars,business_review_count,review_count,yelping_since,average_stars,fans,user_type
0,FQLQXb-Hs-MlbIJf8eXUPw,--Kwhcbkh7jxkhVVQZo2uQ,arQfMJal1tl67Z96ROgPFg,Went for a nice lunch because prices in the Ga...,2014-09-05 21:35:16,4.0,2,Claim Jumper Steakhouse & Bar,Nashville,TN,"Steakhouses, American (Traditional), Restauran...",2.5,347,58,2014-05-30 16:58:11,3.62,2,Power
1,FoiFi2X684BwaYuYDnqBvw,--Kwhcbkh7jxkhVVQZo2uQ,Ld805G25xHALqbBo1Sypbg,It was okay. If you don't know anything about ...,2019-11-11 03:51:53,3.0,1,National Blues Museum,Saint Louis,MO,"Arts & Entertainment, Music Venues, Venues & E...",4.5,45,58,2014-05-30 16:58:11,3.62,2,Power
2,G4YEeMu4Sj1XUEmlJwGe4A,--Kwhcbkh7jxkhVVQZo2uQ,tIvfmgT1qMeAEQf8CI5fPQ,Ate dinner there tonight. Service was pretty f...,2014-09-06 01:09:35,4.0,2,Caney Fork River Valley Grille,Nashville,TN,"Bars, Nightlife, American (Traditional), Barbe...",3.5,574,58,2014-05-30 16:58:11,3.62,2,Power
3,bSz0fCiKRJAB0qI9lB519A,--Kwhcbkh7jxkhVVQZo2uQ,ORL4JE6tz3rJxVqkdKfegA,The place is absolutely huge! That's the attra...,2014-09-07 16:22:03,4.0,2,Gaylord Opryland Resort & Convention Center,Nashville,TN,"Venues & Event Spaces, Performing Arts, Arts &...",3.0,1639,58,2014-05-30 16:58:11,3.62,2,Power
4,QPF7spAqCc-D81GeXcfIoQ,--RJK834fiQXm21VpJp_nw,aIoUwpy5ZFQXUDxWMnMZ4Q,There are new owners here. Way over priced and...,2019-08-25 23:17:52,1.0,0,Pete & Shorty's,Pinellas Park,FL,"Seafood, Diners, Beer, Wine & Spirits, Comfort...",3.5,146,1,2018-02-04 20:34:16,2.50,0,Casual


In [22]:
import glob, shutil

csv_files = glob.glob(os.path.join(PATH_REVIEWS_OUT, "part-*.csv"))

if csv_files:
    shutil.copy(csv_files[0], FINAL_CSV)
    ukuran = os.path.getsize(FINAL_CSV) / (1024**2)
    print(f"Disalin ke: {FINAL_CSV} ({ukuran:.1f} MB)")
else:
    print("ERROR: file part-*.csv tidak ditemukan!")

df_check = pd.read_csv(FINAL_CSV, nrows=3)
print(f"\nKolom : {list(df_check.columns)}")
df_check

Disalin ke: D:\big-data-ai-sentiment\data\processed\reviews_sampled.csv (5424.5 MB)

Kolom : ['review_id', 'user_id', 'business_id', 'text', 'date', 'stars', 'sentiment', 'name', 'city', 'state', 'categories', 'business_stars', 'business_review_count', 'review_count', 'yelping_since', 'average_stars', 'fans', 'user_type']


,review_id,user_id,business_id,text,date,stars,sentiment,name,city,state,categories,business_stars,business_review_count,review_count,yelping_since,average_stars,fans,user_type
0,FQLQXb-Hs-MlbIJf8eXUPw,--Kwhcbkh7jxkhVVQZo2uQ,arQfMJal1tl67Z96ROgPFg,Went for a nice lunch because prices in the Ga...,2014-09-05 21:35:16,4.0,2,Claim Jumper Steakhouse & Bar,Nashville,TN,"Steakhouses, American (Traditional), Restauran...",2.5,347,58,2014-05-30 16:58:11,3.62,2,Power
1,FoiFi2X684BwaYuYDnqBvw,--Kwhcbkh7jxkhVVQZo2uQ,Ld805G25xHALqbBo1Sypbg,It was okay. If you don't know anything about ...,2019-11-11 03:51:53,3.0,1,National Blues Museum,Saint Louis,MO,"Arts & Entertainment, Music Venues, Venues & E...",4.5,45,58,2014-05-30 16:58:11,3.62,2,Power
2,G4YEeMu4Sj1XUEmlJwGe4A,--Kwhcbkh7jxkhVVQZo2uQ,tIvfmgT1qMeAEQf8CI5fPQ,Ate dinner there tonight. Service was pretty f...,2014-09-06 01:09:35,4.0,2,Caney Fork River Valley Grille,Nashville,TN,"Bars, Nightlife, American (Traditional), Barbe...",3.5,574,58,2014-05-30 16:58:11,3.62,2,Power


In [23]:
print("=" * 60)
print("RINGKASAN PIPELINE 01 — SPARK")
print("=" * 60)

for label, path in [
    ("reviews_sampled.csv", FINAL_CSV),
    ("business_clean.parquet", PATH_BUSINESS_OUT),
    ("user_clean.parquet", PATH_USER_OUT),
]:
    if os.path.exists(path):
        if os.path.isdir(path):
            ukuran = sum(
                os.path.getsize(f)
                for f in glob.glob(os.path.join(path, "**"), recursive=True)
                if os.path.isfile(f)
            )
        else:
            ukuran = os.path.getsize(path)
        print(f"  {label:30s} : {ukuran / (1024**2):.1f} MB")
    else:
        print(f"  {label:30s} : FILE TIDAK DITEMUKAN!")

print("\nDistribusi sentimen final:")
df_final.groupBy("sentiment") \
        .agg(
            F.count("*").alias("jumlah"),
            F.round(F.count("*") / df_final.count() * 100, 2).alias("persen_pct")
        ) \
        .orderBy("sentiment") \
        .show()

print("\nPipeline 01 selesai. Lanjut ke 02_eda.ipynb")

RINGKASAN PIPELINE 01 — SPARK
  reviews_sampled.csv            : 5424.5 MB
  business_clean.parquet         : 9.3 MB
  user_clean.parquet             : 71.1 MB

Distribusi sentimen final:
+---------+-------+----------+
|sentiment| jumlah|persen_pct|
+---------+-------+----------+
|        0|1613801|     23.09|
|        1| 691934|       9.9|
|        2|4684545|     67.02|
+---------+-------+----------+


Pipeline 01 selesai. Lanjut ke 02_eda.ipynb


## 10. Tutup SparkSession

Pipeline selesai. SparkSession ditutup supaya RAM dan port dibebaskan.

In [24]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.


## 11. Konversi CSV → Parquet

Spark CSV writer tidak selalu menghasilkan file yang bisa dibaca pandas dengan sempurna kalau ada newline di dalam field `text`. Solusinya: baca ulang dengan Spark (pakai `multiLine=True` dan escape char yang benar), lalu simpan sebagai Parquet. Format ini aman untuk teks apapun dan lebih efisien dibaca pandas.

In [25]:
import os, glob
from pyspark.sql import SparkSession

spark_conv = (
    SparkSession.builder
    .appName("Parquet_Convert")
    .config("spark.driver.memory", "8g")
    .config("spark.local.dir", "D:/spark-temp")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark_conv.sparkContext.setLogLevel("WARN")
print(f"SparkSession: {spark_conv.version}")

PATH_PROCESSED       = r"D:\big-data-ai-sentiment\data\processed"
PATH_REVIEWS_OUT     = os.path.join(PATH_PROCESSED, "_reviews_sampled_spark")
PATH_REVIEWS_PARQUET = os.path.join(PATH_PROCESSED, "reviews_sampled.parquet")

print("Membaca ulang CSV dengan multiLine=True ...")
df_conv = (
    spark_conv.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", "\\")
    .option("inferSchema", "true")
    .csv(PATH_REVIEWS_OUT)
)

jumlah = df_conv.count()
print(f"Baris terbaca  : {jumlah:,}")

print("\nDistribusi sentimen (harus 0/1/2 saja):")
df_conv.groupBy("sentiment").count().orderBy("sentiment").show()

print("Menyimpan ke Parquet ...")
df_conv.coalesce(1).write.mode("overwrite").parquet(PATH_REVIEWS_PARQUET)

ukuran = sum(
    os.path.getsize(f)
    for f in glob.glob(os.path.join(PATH_REVIEWS_PARQUET, "**"), recursive=True)
    if os.path.isfile(f)
)
print(f"Parquet tersimpan : {PATH_REVIEWS_PARQUET}  ({ukuran/1024**2:.1f} MB)")

spark_conv.stop()
print("SparkSession ditutup.")

SparkSession: 3.3.2
Membaca ulang CSV dengan multiLine=True ...
Baris terbaca  : 6,990,280

Distribusi sentimen (harus 0/1/2 saja):
+---------+-------+
|sentiment|  count|
+---------+-------+
|        0|1613801|
|        1| 691934|
|        2|4684545|
+---------+-------+

Menyimpan ke Parquet ...
Parquet tersimpan : D:\big-data-ai-sentiment\data\processed\reviews_sampled.parquet  (3048.4 MB)
SparkSession ditutup.
